# 5-1절 연습 문제 풀이

이 노트북은 5-1절 연습 문제(5-1 ~ 5-3)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 가능하다.

- 본문 예제 코드는 `code_examples/ch05/` 아래 예제 노트북을 참고한다.
- 내려받는 데이터셋은 저장소 규약에 따라 `download/` 디렉터리에 저장한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DOWNLOAD_ROOT = '../../download'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'학습 장치: {device}')

학습 장치: cuda


In [2]:
# 본문 코드 5-2와 같은 방식으로 이미지 전체에 필터를 적용해 특징 지도를 만든다
def make_feature_map(img_tensor, conv_filter):
    k = conv_filter.shape[0]
    h, w = img_tensor.shape
    feature_map = torch.zeros(h - k + 1, w - k + 1)
    for i in range(h - k + 1):
        for j in range(w - k + 1):
            region = img_tensor[i:i + k, j:j + k]
            feature_map[i, j] = (region * conv_filter).sum()
    return feature_map

mnist_test = datasets.MNIST(root=DOWNLOAD_ROOT, train=False, download=True,
                            transform=transforms.ToTensor())
sample = mnist_test[0][0][0]     # (1, 28, 28) -> (28, 28)
print(f'샘플 이미지 형태: {tuple(sample.shape)}, 정답 {mnist_test[0][1]}')

샘플 이미지 형태: (28, 28), 정답 7


## 연습 문제 5-1

> 다음 세 필터로 탐지할 수 있는 특징은 어떤 형태일까?
> - 필터 1: [[-1,0,1],[-1,0,1],[-1,0,1]]
> - 필터 2: [[-1,1,-1],[-1,1,-1],[-1,1,-1]]
> - 필터 3: [[1,-1,1],[1,-1,1],[1,-1,1]]
>
> 필터가 탐지할 수 있는 특징의 형태를 예상한 후, [코드 5-2]를 참고해 숫자마다 세 필터를 사용해 만든
> 특징 지도를 출력해 보자. 그리고 본문의 세로 방향 경계 필터로 만든 특징 지도와 어떤 차이가 있는지 비교해 보자.

### 예상

필터의 각 열이 무엇을 세는지 보면 알 수 있다. 세 필터 모두 **세로 방향**으로 같은 값이 반복되므로,
세로로 이어지는 패턴에 반응한다. 차이는 가로 방향의 부호 배치에 있다.

| 필터 | 열 구성 | 반응하는 형태 |
|---|---|---|
| 본문 필터 `[1, 0, -1]` | 왼쪽 +, 오른쪽 − | **왼쪽이 밝고 오른쪽이 어두운** 세로 경계 |
| 필터 1 `[-1, 0, 1]` | 왼쪽 −, 오른쪽 + | **왼쪽이 어둡고 오른쪽이 밝은** 세로 경계(본문 필터의 반대) |
| 필터 2 `[-1, 1, -1]` | 가운데만 + | **가운데만 밝은 세로 선**(양쪽이 어두운 밝은 획) |
| 필터 3 `[1, -1, 1]` | 가운데만 − | **가운데만 어두운 세로 선**(필터 2의 반대) |

즉 필터 1은 경계의 방향이 반대인 검출기, 필터 2와 3은 경계가 아니라 **선 자체**를 찾는 검출기다.

In [3]:
FILTERS = {
    '본문 필터 (왼쪽 밝음)': torch.tensor([[1., 0., -1.], [1., 0., -1.], [1., 0., -1.]]),
    '필터 1 (오른쪽 밝음)': torch.tensor([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]]),
    '필터 2 (밝은 세로선)': torch.tensor([[-1., 1., -1.], [-1., 1., -1.], [-1., 1., -1.]]),
    '필터 3 (어두운 세로선)': torch.tensor([[1., -1., 1.], [1., -1., 1.], [1., -1., 1.]]),
}

# 특징 지도를 숫자로 요약해 비교한다(시각화는 깃허브 예제 노트북 참고)
print(f'{"필터":>22} {"최댓값":>8} {"최솟값":>8} {"양수 비율":>9} {"최대 위치(행, 열)":>16}')
print('-' * 70)
for name, conv_filter in FILTERS.items():
    fmap = make_feature_map(sample, conv_filter)
    pos_ratio = (fmap > 0.5).float().mean().item() * 100
    idx = fmap.argmax().item()
    print(f'{name:>22} {fmap.max():8.2f} {fmap.min():8.2f} {pos_ratio:8.1f}% '
          f'{str((idx // fmap.shape[1], idx % fmap.shape[1])):>16}')

                    필터      최댓값      최솟값     양수 비율      최대 위치(행, 열)
----------------------------------------------------------------------
         본문 필터 (왼쪽 밝음)     2.98    -2.94      8.0%         (22, 13)
         필터 1 (오른쪽 밝음)     2.94    -2.98      8.6%          (24, 9)
         필터 2 (밝은 세로선)     0.14    -2.93      0.0%         (10, 11)
        필터 3 (어두운 세로선)     2.93    -0.14     23.4%          (8, 17)


### 풀이 해설

숫자로 확인해 보면 예상한 방향이 맞는다는 것을 알 수 있다. 다만 필터 2와 필터 3은 조금 더 들여다볼 필요가 있다.

- **본문 필터와 필터 1은 최댓값과 최솟값의 부호가 뒤집혀 있다.** 같은 경계를 정반대로 본다는 뜻이다.
  최댓값이 나타나는 위치도 서로 다른데(각각 (22, 13)과 (24, 9)), 획의 왼쪽 경계와 오른쪽 경계를 각각 찾기 때문이다.
- **필터 2는 거의 음수만 출력한다.** 최댓값이 0.14로 0에 가깝고 최솟값은 -2.93이다.
  필터 2가 찾는 것은 **폭이 1픽셀인 밝은 세로선**인데, MNIST의 획은 두께가 2~3픽셀이라 그런 선이 거의 없다.
  오히려 두꺼운 획 위에서는 세 열이 모두 밝아 `-1 + 1 - 1 = -1`이 되므로 큰 음수가 나온다.
- **필터 3은 필터 2의 부호를 뒤집은 것**이므로 정확히 반대로 나온다(최댓값 2.93, 최솟값 -0.14).
  두꺼운 획 위에서 큰 양수가 나오니, 결과만 보면 '획의 몸통 탐지기'처럼 동작한다.

여기서 두 가지를 알 수 있다. 첫째, **필터의 부호 배치가 곧 탐지 대상**이다.
둘째, **필터가 찾도록 설계된 특징이 데이터에 없으면 그 필터는 쓸모가 없다.**
필터 2는 설계 자체는 멀쩡하지만 MNIST에서는 유용한 정보를 거의 만들어 내지 못한다.
본문 p9가 "필터의 요솟값이 학습 과정에서 최적화되는 파라미터"라고 한 이유가 여기에 있다.
사람이 고른 필터가 항상 그 데이터에 맞는다는 보장이 없으므로, 데이터에 맞는 부호 배치를 모델이 직접 찾게 하는 것이다.

### 문제 검토

- **적절성: 적합.** 본문이 필터 하나만 보여 준 뒤 "모델이 필터를 학습한다"로 넘어가는데,
  이 문제가 그 사이를 메운다. 네 필터를 비교하면 부호 배치가 탐지 대상을 결정한다는 것이 드러난다.
- **[검토] 세 필터의 선택이 좋다.** 필터 1은 본문 필터의 반대, 필터 2와 3은 서로 반대로 짝을 이뤄
  '반대 부호 = 반대 특징'이라는 규칙을 두 번 확인하게 한다.
- **[검토] 무엇을 비교할지 한 걸음 더 짚어 주면 좋다.** "어떤 차이가 있는지 비교해 보자"만으로는
  그림을 눈으로 훑고 넘어가기 쉽다. 최댓값과 최솟값의 부호를 견주어 보라고 하면 관찰이 분명해진다.

## 연습 문제 5-2

> 가로 방향 경계선과 대각선 방향 경계선을 탐지하는 필터를 직접 설계하고,
> [코드 5-2]를 참고해 MNIST 데이터셋 샘플의 특징 지도를 출력해 보자.

In [4]:
# 세로 경계 필터를 90도 돌리면 가로 경계 필터가 된다
MY_FILTERS = {
    '가로 경계 (위 밝음)': torch.tensor([[1., 1., 1.], [0., 0., 0.], [-1., -1., -1.]]),
    '대각선 경계 (↘ 방향)': torch.tensor([[0., 1., 1.], [-1., 0., 1.], [-1., -1., 0.]]),
    '대각선 경계 (↗ 방향)': torch.tensor([[1., 1., 0.], [1., 0., -1.], [0., -1., -1.]]),
}

print(f'{"필터":>20} {"최댓값":>8} {"최솟값":>8} {"강한 반응 비율":>13}')
print('-' * 56)
for name, conv_filter in {**FILTERS, **MY_FILTERS}.items():
    fmap = make_feature_map(sample, conv_filter)
    strong = (fmap.abs() > 1.0).float().mean().item() * 100
    print(f'{name:>20} {fmap.max():8.2f} {fmap.min():8.2f} {strong:12.1f}%')

                  필터      최댓값      최솟값      강한 반응 비율
--------------------------------------------------------
       본문 필터 (왼쪽 밝음)     2.98    -2.94         12.7%
       필터 1 (오른쪽 밝음)     2.94    -2.98         12.7%
       필터 2 (밝은 세로선)     0.14    -2.93         16.1%
      필터 3 (어두운 세로선)     2.93    -0.14         16.1%
        가로 경계 (위 밝음)     2.99    -2.99         14.5%
       대각선 경계 (↘ 방향)     2.47    -2.56          9.8%
       대각선 경계 (↗ 방향)     2.92    -2.90         17.6%


### 풀이 해설

설계 요령은 간단하다. **본문의 세로 경계 필터를 90도 돌리면 가로 경계 필터**가 되고,
대각선 방향으로 부호를 배치하면 대각선 경계 필터가 된다.

핵심은 **필터의 합이 0이 되도록 만드는 것**이다. 합이 0이면 밝기가 균일한 영역에서 결과가 0이 되어,
밝기 자체가 아니라 **밝기의 변화**에만 반응한다. 위 필터들은 모두 요소의 합이 0이다.
본문 p7이 "부분 이미지 영역이 모두 같은 밝기의 픽셀이면 합성곱 연산의 결과는 0"이라고 한 것이 이 성질이다.

샘플(숫자 7)에서 결과를 보면 가로 경계 필터의 반응이 가장 크다(최댓값 2.99, 최솟값 -2.99).
7의 윗변이 긴 가로획이니 당연한 결과다. 대각선 필터 중에서는 ↗ 방향(최댓값 2.92)이 ↘ 방향(2.47)보다 강한데,
7의 삐침이 오른쪽 위에서 왼쪽 아래로 내려가는 방향이기 때문이다.

**숫자마다 잘 반응하는 필터가 다르다**는 점이 중요하다. 그래서 합성곱 계층은 필터를 하나만 두지 않고
여러 개를 두어 서로 다른 방향의 경계를 동시에 찾는다.

### 문제 검토

- **적절성: 적합.** 5-1이 주어진 필터를 해석하는 문제라면 이 문제는 직접 설계하는 문제다. 난도가 자연스럽게 오른다.
- **[검토] 설계 기준이 없다.** '탐지하는 필터를 설계하라'만으로는 무엇을 만족해야 올바른 필터인지 알 수 없다.
  실제로 가장 중요한 조건은 **필터 요소의 합이 0이어야 한다**는 것인데(그래야 균일한 영역에서 0이 나온다),
  본문 p7이 그 성질을 언급하고 있으므로 지문에서 한 번 상기시키면 독자가 방향을 잡는다.
- **[검토] 결과 확인 방법.** 5-1과 마찬가지로 시각화가 필요하다는 안내가 있으면 좋다.

**윤문안**

> **5-2**. 가로 방향 경계선과 대각선 방향 경계선을 탐지하는 필터를 직접 설계하고, [코드 5-2]를 참고해
> MNIST 데이터셋 샘플의 특징 지도를 출력해 보자. 설계한 필터가 밝기가 균일한 영역에서 0을 출력하는지도 확인해 보자.

## 연습 문제 5-3

> 다음에 제시된 조건이 변화했을 때 합성곱 신경망의 파라미터 수가 어떻게 바뀌는지 설명해 보자.
> - 특징 탐지, 요약, 조합 과정의 반복 횟수를 늘렸을 때
> - 합성곱 필터의 크기를 늘렸을 때
> - 풀링의 커널 크기를 늘렸을 때
> - 합성곱 필터의 수를 늘렸을 때

In [5]:
def build_cnn(n_blocks=2, kernel=3, pool=2, channels=(16, 32)):
    layers, in_ch, size = [], 1, 28
    for i in range(n_blocks):
        out_ch = channels[min(i, len(channels) - 1)] * (2 ** max(0, i - len(channels) + 1))
        layers += [nn.Conv2d(in_ch, out_ch, kernel_size=kernel, padding=kernel // 2),
                   nn.ReLU(), nn.MaxPool2d(kernel_size=pool)]
        in_ch, size = out_ch, size // pool
    layers += [nn.Flatten(), nn.Linear(in_ch * size * size, 10)]
    return nn.Sequential(*layers)

def count(model):
    conv = sum(p.numel() for m in model if isinstance(m, nn.Conv2d) for p in m.parameters())
    fc = sum(p.numel() for m in model if isinstance(m, nn.Linear) for p in m.parameters())
    return conv, fc, conv + fc

CASES = [
    ('기준 (블록 2, 필터 3x3, 풀링 2)', dict()),
    ('반복 횟수 2 -> 3', dict(n_blocks=3)),
    ('필터 크기 3 -> 5', dict(kernel=5)),
    ('풀링 커널 2 -> 4', dict(pool=4)),
    ('필터 수 (16,32) -> (32,64)', dict(channels=(32, 64))),
]
print(f'{"조건":>28} {"합성곱":>10} {"완전 연결":>11} {"합계":>10}')
print('-' * 64)
for name, kwargs in CASES:
    conv, fc, total = count(build_cnn(**kwargs))
    print(f'{name:>28} {conv:10,d} {fc:11,d} {total:10,d}')

                          조건        합성곱       완전 연결         합계
----------------------------------------------------------------
     기준 (블록 2, 필터 3x3, 풀링 2)      4,800      15,690     20,490
                반복 횟수 2 -> 3     23,296       5,770     29,066
                필터 크기 3 -> 5     13,248      15,690     28,938
                풀링 커널 2 -> 4      4,800         330      5,130
     필터 수 (16,32) -> (32,64)     18,816      31,370     50,186


### 풀이 해설

네 조건이 파라미터에 미치는 영향이 **서로 다른 방식**이라는 점이 이 문제의 핵심이다.
합성곱 계층과 완전 연결 계층을 나누어 보면 분명해진다.

**1. 반복 횟수를 늘리면(2 → 3)** 합성곱 계층이 하나 더 생겨 합성곱 파라미터가 4,800에서 23,296으로 크게 는다.
동시에 풀링이 한 번 더 일어나 특징 지도가 작아지므로 **완전 연결 계층은 15,690에서 5,770으로 줄어든다.**
두 변화가 반대 방향이고, 여기서는 합성곱의 증가폭이 더 커서 전체는 20,490에서 29,066으로 늘었다.
어느 쪽이 이길지는 채널 수를 어떻게 늘리느냐에 달렸으므로, **반복 횟수만 보고 증감을 단정할 수 없다.**

**2. 필터 크기를 늘리면(3×3 → 5×5)** 필터 하나의 요소 수가 k²에 비례해 9에서 25로, 약 2.8배가 된다.
실제로 합성곱 파라미터가 4,800에서 13,248로 늘었다. 패딩으로 크기를 유지했으므로 **완전 연결 계층은 15,690 그대로다.**

**3. 풀링 커널을 늘리면(2 → 4) 파라미터는 늘지 않는다.** 풀링은 학습 파라미터가 없는 계층이므로
합성곱 파라미터는 4,800으로 그대로다. 대신 특징 지도가 훨씬 작아져 **완전 연결 계층이 15,690에서 330으로 급감**하고,
전체는 20,490에서 5,130으로 4분의 1 수준이 된다. 네 조건 중 유일하게 파라미터가 줄어드는 경우다.

**4. 필터 수를 늘리면(16, 32 → 32, 64)** 합성곱 파라미터가 늘고(4,800 → 18,816), 특징 지도의 개수가 늘어
**완전 연결 계층의 입력도 함께 늘어난다**(15,690 → 31,370). 두 곳이 동시에 늘어나므로 증가폭이 가장 크다.

정리하면 **풀링만 파라미터를 직접 늘리지 않으며, 필터 수는 두 곳에 동시에 영향을 준다.**
그리고 이 문제에서 가장 배울 점은, 합성곱 신경망의 파라미터 대부분이 **완전 연결 계층에 몰려 있을 수 있다**는 사실이다.
기준 모델에서도 전체의 76%가 완전 연결 계층 몫이다.

### 문제 검토

- **적절성: 적합. 잘 설계된 문제다.** 네 조건이 각각 다른 방식으로 작용하도록 골랐다.
  특히 풀링 커널(파라미터 없음)과 필터 수(두 곳에 영향)를 함께 둔 덕분에, 합성곱 신경망의 파라미터가
  어디에서 생기는지 정확히 이해했는지 확인할 수 있다.
- **[검토] 완전 연결 계층까지 포함하는지 분명히 하면 좋다.** '합성곱 신경망의 파라미터 수'라고만 해서,
  독자가 합성곱 계층만 세면 세 번째 조건(풀링)의 답이 '변화 없음'이 되어 버린다.
  실제로는 완전 연결 계층까지 봐야 '풀링을 키우면 전체 파라미터가 준다'는 관찰에 이른다.
- **[검토] 계산해 확인하라는 안내가 있으면 좋다.** 설명만 요구하므로 머릿속으로 끝낼 수 있다.
  `torchinfo.summary()`나 `numel()`로 실제 수를 세어 보면 예상과 맞는지 확인할 수 있다.

**윤문안**

> **5-3**. 다음에 제시된 조건이 변화했을 때 합성곱 신경망의 파라미터 수가 어떻게 바뀌는지, 합성곱 계층과
> 완전 연결 계층으로 나누어 설명해 보자. 그리고 실제로 모델을 만들어 파라미터 수를 세어 예상과 비교해 보자.
> (네 조건은 그대로)